### Mugrade boilerplate


In [ ]:
### Run this cell to install and import the homework tests
%pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part2_llm_arch_tests.py

import mugrade
import os
from part2_llm_arch_tests import *
os.environ["MUGRADE_HW"] = "Part 2 - LLM Architecture"
# Read the Mugrade API key from the environment or from a local .env file.
# Never hard-code the key. See .env.example in the repository root.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
if not os.environ.get("MUGRADE_KEY"):
    print("MUGRADE_KEY is not set. Cells that use @mugrade.submit_tests need it.")

### LLM Transformer Architecture

Implement the LLM architecture presented in class.  This will _only_ be the architecture portion in this part of the assignment, and the training will be done in the next portion (released after the next lecture).

Note that there was error with the version typed live in class, and `self.params` was replaced by `llm.params` within the class in several instaces.  Internal reference to parameters (or the `buffer`) within the class should also be done via `self.params`.  You should _also_ use `[-sin, sin]` for the second RoPE weight, rather than the `[sin, -sin]` written in class, in order to match the standard convention.


In [2]:
import torch
import math

@mugrade.submit_tests
def embedding(x, weights, dtype):
    ### BEGIN YOUR CODE
    return weights[x].to(dtype=dtype)
    ### END YOUR CODE

@mugrade.submit_tests
def linear(x, weights):
    ### BEGIN YOUR CODE
    return x @ weights.type_as(x)
    ### END YOUR CODE


@mugrade.submit_tests
def silu(x):
    ### BEGIN YOUR CODE
    return x / (1 + torch.exp(-x))
    ### END YOUR CODE

@mugrade.submit_tests
def rms_norm(x):
    ### BEGIN YOUR CODE
    x_typed = x.float()
    x_normed = x_typed / (
        (x_typed**2)
            .mean(
                dim = -1,
                keepdim = True,
            )
            .sqrt()
    )
    return x_normed.type_as(x)
    ### END YOUR CODE

@mugrade.submit_tests
def softmax(x):
    ### BEGIN YOUR CODE
    x_numerical_stable = \
        x - x.max(dim = -1, keepdim=True).values
    x_exp = x_numerical_stable.exp()
    return x_exp / (
        x_exp.sum(dim=-1, keepdim=True)
    )
    ### END YOUR CODE

@mugrade.submit_tests
def self_attn(q,k,v,mask):
    ### BEGIN YOUR CODE
    attention_score = softmax(
        q @ k.transpose(-2, -1) / math.sqrt(k.size(-1)) + mask
    )
    return attention_score @ v
    ### END YOUR CODE

class LLM:
    @mugrade.submit_tests
    def rope(self, x):
        ### BEGIN YOUR CODE
        return (
            (self.buffers["rope_w1"] * x) + 
            (self.buffers["rope_w2"] * x.reshape(-1, 
                2).flip(-1).reshape_as(x))
        ).type_as(x)
        ### END YOUR CODE

    @mugrade.submit_tests
    def multihead_attn(self, x, layer, mask):
        ### BEGIN YOUR CODE
        
        B, L, D = x.shape
        q = linear(x, self.params[f"wq_{layer}"])
        k = linear(x, self.params[f"wk_{layer}"])
        v = linear(x, self.params[f"wv_{layer}"])
        wp = self.params[f"wp_{layer}"]

        q = q.reshape(
            B, L, self.num_heads, D // self.num_heads
        ).transpose(1, 2)
        k = k.reshape(
            B, L, self.num_heads, D // self.num_heads
        ).transpose(1, 2)
        v = v.reshape(
            B, L, self.num_heads, D // self.num_heads
        ).transpose(1, 2)

        q, k = rms_norm(self.rope(q)), rms_norm(self.rope(k))

        attn_output = self_attn(q,k,v,mask)

        return linear(
            attn_output.transpose(1, 2).reshape_as(x),
            wp
        )

        ### END YOUR CODE

    @mugrade.submit_tests
    def mlp(self, x, layer):
        ### BEGIN YOUR CODE
        return linear(
            silu(
                linear(
                    x,
                    self.params[f"w1_{layer}"]
                )
            ),
            self.params[f"w2_{layer}"]
        )
        ### END YOUR CODE

    @mugrade.submit_tests
    def transformer_block(self, x, layer, mask):
        ### BEGIN YOUR CODE
        x = x + \
            self.multihead_attn(
                rms_norm(x), layer, mask
            )
        return x + self.mlp(
            rms_norm(x), layer
        )
        ### END YOUR CODE

    @mugrade.submit_tests
    def __init__(self, config):
        ### BEGIN YOUR CODE
        self.num_layers = config["depth"]
        self.dtype = config["dtype"]

        vocab_size = config["vocab_size"]
        sequence_len = config["seq_len"]

        hidden_dim = self.num_layers * config["aspect_ratio"]
        self.mlp_dim = hidden_dim * config["mlp_multiple"]

        self.head_dim = config["head_dim"]
        self.num_heads = hidden_dim // self.head_dim

        self.params = {}
        dim_scale, dim_scale_mlp = \
            math.sqrt(2/hidden_dim), math.sqrt(2/self.mlp_dim)

        self.params["embedding"] = torch.randn(
            vocab_size, hidden_dim
        ) * dim_scale
        self.params["output"] = torch.randn(
            hidden_dim, vocab_size
        ) * dim_scale

        for i in range(self.num_layers):
            self.params[f"wq_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wk_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wv_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wp_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"w1_{i}"] = \
                torch.randn(
                    hidden_dim, self.mlp_dim
                ) * dim_scale
            self.params[f"w2_{i}"] = \
                torch.randn(
                    self.mlp_dim, hidden_dim
                ) * dim_scale_mlp

        self.buffers = {}
        mask = torch.full(
            (sequence_len, sequence_len),
            -float("inf"))
        self.buffers["mask"] = \
            torch.triu(mask, diagonal=1)


        freqs = config["rope_theta"] ** (
            (-2 * torch.arange(
                self.head_dim // 2
            ) / self.head_dim)
        )
        theta = torch.outer(
            torch.arange(sequence_len),
            freqs
        )

        cos, sin = torch.cos(theta), torch.sin(theta)
        self.buffers["rope_w1"] = \
            torch.stack([cos, cos], dim = -1).reshape(
                sequence_len, self.head_dim
            )
        self.buffers["rope_w2"] = \
            torch.stack([-sin, sin], dim = -1).reshape(
                sequence_len, self.head_dim
            )

        ### END YOUR CODE

    @mugrade.submit_tests
    def __call__(self, tokens):
        ### BEGIN YOUR CODE
        x = rms_norm(
            embedding(
                tokens,
                self.params["embedding"],
                self.dtype
            )
        )

        for i in range(self.num_layers):
            x = self.transformer_block(
                x, i, self.buffers["mask"]
            )

        x_final = \
            rms_norm(x)

        return linear(
            x_final, 
            self.params["output"]
        )
        ### END YOUR CODE

    @mugrade.submit_tests
    def save(self, filename):
        ### BEGIN YOUR CODE
        torch.save(self.params, filename)
        ### END YOUR CODE

    @mugrade.submit_tests
    def load(self, filename):
        ### BEGIN YOUR CODE
        self.params = \
            torch.load(filename, weights_only=True)
        ### END YOUR CODE


Mugrade: Submitting tests for function embedding():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function linear():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function silu():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function rms_norm():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function softmax():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function self_attn():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function rope():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function multihead_attn():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function mlp():
Grader test 1 passed
Grader test 2 passed
Gra

The tests for the code above will all pass based upon specific instances from the test cases, but if you want to try out the construction on a realistic (but still very small) model class, you can try out the following model, using the code to load the configuration from class.

In [ ]:
%%writefile config.d12.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "head_dim": 128,
    "mlp_multiple": 4,
    "vocab_size": 32768,
    "seq_len": 2048,
    "num_gpus": 1,
    "batch_size": 32,
    "token_multiple": 20,
    "rope_theta": 10000,
    "dtype": "bfloat16",
    "lr": 1e-3,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "tokenizer": "tokenizer_50M.bpe"
}